# Project - First Draft
this file is for experimenting, writing first bits of code and testing

In [ ]:

# import important packages
import requests
import pandas as pd
import geopandas as gpd
import time
import matplotlib.pyplot as plt
import folium
import contextily
import cmcrameri


from cartopy import crs as ccrs
from geodatasets import get_path

c:\Users\isabe\miniconda3\envs\sds-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# test my MAP_KEY

map_key = "ea49e5fe6beaf3a00954c71727386596"

import pandas as pd
import requests
key_url = f"https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?map_key={map_key}"
try:
  response = requests.get(key_url)

  if response.status_code == 200:
    data = response.json()
    df = pd.Series(data)
    display(df)
  else:
    print(f"Error in the query: HTTP {response.status_code}")
    print(response.text)

  
except Exception as e:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print (f"There is an issue with the query: {e}\n try in your browser: {key_url}")
  


transaction_limit             5000
current_transactions             0
transaction_interval    10 minutes
dtype: object

### Query API and Import Data

In [14]:
import requests
import time
import datetime

# Define Dates: start and enddate format (YYYY-MM-DD), oldest date 2017?
startdate = "2023-10-22"
enddate = "2023-10-26"


type(startdate)
type(enddate)

start = pd.to_datetime(startdate)
end = pd.to_datetime(enddate)

difference = end-start
print(difference)

days = (end-start).days
print (days)


4 days 00:00:00
4


#### first try to load data

In [ ]:

import requests
import time
import datetime
import geopandas as gpd
import pandas as pd

map_key = "ea49e5fe6beaf3a00954c71727386596"

# Define Dates: start and enddate format (YYYY-MM-DD), oldest date 2017?
startdate = "2023-10-22"
enddate = "2023-10-26"

# calculate days out of dates --> FIRMS uses a number of days in the API-call
start = pd.to_datetime(startdate)
end = pd.to_datetime(enddate)

days = (end-start).days

# transform dates to FIRMS accepted format (YYYY-MM-DD)

# for loop später

#area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/world/1'

#date.today() fetches the current date -> is a date-object
# srftime converts dateobject to strings as FIRMS needs it in a string format (YYYY-MM-DD)

# source for this???


# define coordinates for South America

region_coords = "-55.0,-81.5,12.5,-35.0"

# define sensor and products
sensor = "VIIRS_NOAA20_SP"
product = "fire"

# define API endpoint url

api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{sensor}/{region_coords}/{days}"


# define query parameters
parameters = {
    "key": map_key,
    "start_date": startdate,
    "end_date": enddate,
    "area": region_coords,
    "sensor": sensor,
    "product": product,
    "format": "csv"    
}


# fetch the data

response = requests.get(api_url, params = parameters)

# check the status and extract the data
if response.status_code == 200: 
    print("Request successful\n")

    # read url and create dataframe
    df_fire = gpd.read_file(api_url)
    print(f"\n Data download successful! {len(df_fire)} fires found")  ## NA handling??
    display(df_fire.head(6))

    # option: include a saving option if wanted
    #df_fire.to_csv("firms_data_startdate_enddate.csv", index=False) wieso index = false??

elif response.status_code == 404:
    print(f"Error 404: Page not found")

elif response.status_code == 401:
    print(f"Error 401: Invalid API-Key, please check map-key")

elif response.status_code == 400:
    print(f"Error 400: Wrong parameters, please check input parameters.")


else: 
    print(f"Request failed: Status {response.status_code}")
    print(response.text)


Error 400: Wrong parameters, please check input parameters.


#### second try to load data; works but only fetches the last 8 days

current FIRMS-URL fetches only the data for the last 8 days. It does not consider start and enddate as I calculated it. Have to create a workaround so api accepts it

In [ ]:

import requests
import time
import datetime 
import geopandas as gpd
import pandas as pd

map_key = "ea49e5fe6beaf3a00954c71727386596"

# Define Dates: start and enddate format (YYYY-MM-DD), oldest date 2017?
startdate = "2024-09-22"
enddate = "2024-09-30"

# calculate days out of dates --> FIRMS uses a number of days in the API-call
#start = pd.to_datetime(startdate)
#end = pd.to_datetime(enddate)

#days = (end-start).days 

# transform dates to FIRMS accepted format (YYYY-MM-DD)

# for loop später

#area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/world/1'

#date.today() fetches the current date -> is a date-object
# srftime converts dateobject to strings as FIRMS needs it in a string format (YYYY-MM-DD)

# source for this???


# define coordinates for South America

region_coords = "-55.0,-81.5,12.5,-35.0"

# define sensor and products
sensor = "VIIRS_NOAA20_SP"
product = "fire"

# define API endpoint url

api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{sensor}/{region_coords}/{startdate}/{enddate}"



# fetch the data

response = requests.get(api_url)

# check the status and extract the data
if response.status_code == 200: 
    print("Request successful\n")

    # read url and create dataframe
    df_fire = pd.read_csv(api_url)
    print(f"\n Data download successful! {len(df_fire)} fires found")  ## NA handling??
    display(df_fire.head(6))

    # option: include a saving option if wanted
    #df_fire.to_csv("firms_data_startdate_enddate.csv", index=False) wieso index = false??

elif response.status_code == 404:
    print(f"Error 404: Page not found")

elif response.status_code == 401:
    print(f"Error 401: Invalid API-Key, please check map-key")

elif response.status_code == 400:
    print(f"Error 400: Wrong parameters, please check input parameters.")


else: 
    print(f"Request failed: Status {response.status_code}")
    print(response.text)


Error 400: Wrong parameters, please check input parameters.


Area API does not support Bounding Box, only Bounding Box and last x days works -> write work around with a loop (used help of AI)

In [ ]:
import requests
import pandas as pd
import geopandas as gpd
from datetime import timedelta

map_key = "ea49e5fe6beaf3a00954c71727386596"

# User Input Parameters -> can be changed for different queries

startdate = "2024-09-22"
enddate = "2024-09-30"

sensor = "VIIRS_NOAA20_SP"

# define coordinates for Bounding Box around South America (lon/lat)
region_coords = "-81.5,-55.0,-35.0,12.5"


# date conversion to python/pandas format
start = pd.to_datetime(startdate)
end = pd.to_datetime(enddate)



# define parameters
step_days = 


In [ ]:
#lumo

import pandas as pd
import requests
from datetime import date

# --- KONFIGURATION ---
API_KEY = "ea49e5fe6beaf3a00954c71727386596"  # Ersetze dies mit deinem gültigen Key
BASE_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/"

# 1. Datum festlegen (Heute)
# FIRMS benötigt das Format YYYY-MM-DD
today = date.today().strftime("%Y-%m-%d")

# 2. Region definieren (Beispiel: Ein Rechteck über Deutschland)
# Format: lat_min,lon_min,lat_max,lon_max
# Oder als GeoJSON Polygon-String
region_coords = "47.0,5.0,55.0,15.0"  # Beispiel: Südwest bis Nordost Deutschland

# 3. Sensor und Produkt wählen
# Sensoren: 'VIIRS', 'MODIS', 'SNPP'
# Produkte: 'fire' (Feuer), 'thermal_anomalies' (Thermische Anomalien)
sensor = "VIIRS"
product = "fire"

# --- API AUFRUF ---
params = {
    "key": API_KEY,
    "date": today,
    "area": region_coords,
    "sensor": sensor,
    "product": product,
    "format": "csv"  # FIRMS liefert oft CSV zurück, das ist einfacher zu parsen
}

print(f"Abfrage für Datum: {today}, Region: {region_coords}")

try:
    response = requests.get(BASE_URL, params=params)
    
    # Statuscode prüfen
    if response.status_code == 200:
        # Die API gibt oft CSV zurück, nicht JSON
        # Wir nutzen pandas, um das CSV direkt zu lesen
        df = pd.read_csv(pd.io.common.StringIO(response.text))
        
        print(f"\nErfolgreich! {len(df)} Feuerstellen gefunden.")
        display(df.head()) # Zeige die ersten 5 Zeilen
        
        # Optional: Speichern
        # df.to_csv("firms_data_today.csv", index=False)
        
    elif response.status_code == 401:
        print("Fehler 401: Ungültiger API-Key. Bitte überprüfe deinen Key.")
    elif response.status_code == 400:
        print("Fehler 400: Falsche Parameter. Prüfe Datum und Koordinaten.")
        print(f"URL: {response.url}")
        print(f"Inhalt: {response.text[:200]}")
    else:
        print(f"Fehler: Status {response.status_code}")
        print(response.text)

except Exception as e:
    print(f"Ein unerwarteter Fehler trat auf: {e}")

Abfrage für Datum: 2026-05-07, Region: 47.0,5.0,55.0,15.0
Ein unerwarteter Fehler trat auf: Error tokenizing data. C error: Expected 1 fields in line 7, saw 13



In [ ]:
import pandas as pd
import geopandas as gpd
import requests
import time

# =====================================
# USER INPUT
# =====================================


startdate = "2024-01-01"
enddate   = "2024-01-10"

sensor = "VIIRS_NOAA20_SP"

# South America bounding box
# min_lon, min_lat, max_lon, max_lat
region_coords = "-81.5,-55.0,-35.0,12.5"

# =====================================
# DATE HANDLING
# =====================================

start = pd.to_datetime(startdate)
end = pd.to_datetime(enddate)

date_range = pd.date_range(start, end)

# =====================================
# DOWNLOAD LOOP
# =====================================

all_data = []

for current_date in date_range:

    date_str = current_date.strftime("%Y-%m-%d")

    print(f"Fetching data for {date_str}")

    # FIRMS endpoint
    url = (
        f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{map_key}/{sensor}/{region_coords}/1"
    )



    try:
        # download csv
        df = pd.read_csv(url)

        # add query date
        df["query_date"] = date_str

        # keep only rows from desired date
        if "acq_date" in df.columns:
            df = df[df["acq_date"] == date_str]

        # append if data exists
        if len(df) > 0:
            all_data.append(df)

        # optional: avoid rate limits
        time.sleep(0.2)

    except Exception as e:
        print(f"Error on {date_str}: {e}")

# =====================================
# MERGE RESULTS
# =====================================

if len(all_data) > 0:

    df_fire = pd.concat(all_data, ignore_index=True)

    print(f"\nTotal fire detections: {len(df_fire)}")

    # Convert to GeoDataFrame
    gdf_fire = gpd.GeoDataFrame(
        df_fire,
        geometry=gpd.points_from_xy(
            df_fire.longitude,
            df_fire.latitude
        ),
        crs="EPSG:4326"
    )

    display(gdf_fire.head())

else:
    print("No data found.")

Fetching data for 2024-01-01
Fetching data for 2024-01-02
Fetching data for 2024-01-03
Fetching data for 2024-01-04
Fetching data for 2024-01-05
Fetching data for 2024-01-06
Fetching data for 2024-01-07
Fetching data for 2024-01-08
Fetching data for 2024-01-09
Fetching data for 2024-01-10
No data found.


In [26]:
import requests
import pandas as pd
import geopandas as gpd
import time

# =====================================
# USER INPUT
# =====================================



startdate = "2024-03-01"
enddate   = "2024-03-06"

sensor = "VIIRS_NOAA20_SP"

# South America bounding box
# min_lon, min_lat, max_lon, max_lat
region_coords = "-81.5,-55.0,-35.0,12.5"

# =====================================
# DATE HANDLING
# =====================================

date_range = pd.date_range(startdate, enddate)

# =====================================
# STORAGE
# =====================================

all_data = []

# =====================================
# LOOP
# =====================================

for current_date in date_range:

    date_str = current_date.strftime("%Y-%m-%d")

    print(f"\nFetching data for {date_str}")

    # FIRMS URL
    url = (
        f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{map_key}/{sensor}/{region_coords}/1"
    )

    # request
    response = requests.get(url)

    # =====================================
    # SUCCESS
    # =====================================

    if response.status_code == 200:

        print("Request successful")

        try:
            # convert response to dataframe
            df = pd.read_csv(url)

            # keep only desired day
            if "acq_date" in df.columns:
                df = df[df["acq_date"] == date_str]

            # save query date
            df["query_date"] = date_str

            # append if data exists
            if len(df) > 0:
                all_data.append(df)
                print(f"{len(df)} fires found")

            else:
                print("No fires found")

        except Exception as e:
            print(f"Data processing error: {e}")

    # =====================================
    # ERROR HANDLING
    # =====================================

    elif response.status_code == 400:
        print("Error 400: Wrong parameters")

    elif response.status_code == 401:
        print("Error 401: Invalid API key")

    elif response.status_code == 404:
        print("Error 404: Endpoint not found")

    else:
        print(f"Request failed: {response.status_code}")

    # avoid rate limits
    time.sleep(0.2)

# =====================================
# MERGE RESULTS
# =====================================

if len(all_data) > 0:

    df_fire = pd.concat(all_data, ignore_index=True)

    print(f"\nTotal fires downloaded: {len(df_fire)}")

    # convert to GeoDataFrame
    gdf_fire = gpd.GeoDataFrame(
        df_fire,
        geometry=gpd.points_from_xy(
            df_fire.longitude,
            df_fire.latitude
        ),
        crs="EPSG:4326"
    )

    display(gdf_fire.head())

else:
    print("\nNo data collected.")


Fetching data for 2024-03-01
Request successful
No fires found

Fetching data for 2024-03-02
Request successful
No fires found

Fetching data for 2024-03-03
Request successful
No fires found

Fetching data for 2024-03-04
Request successful
No fires found

Fetching data for 2024-03-05
Request successful
No fires found

Fetching data for 2024-03-06
Request successful
No fires found

No data collected.


currently there are many fires in south america, but my code does not show it, even though the code above fetches the data from the past day

So new game plan; try to get fire data for the past 7 days, try to visualize this, then download historical fire data because with the current api historical data cannot be accessed

data older than 7 days must be accessed via FIRMS Archive Download 

then, write code which should display the exact time range chosen for the current data but 1-5 years ago -> comparison 

maybe change satellite when VIIRS does not display it

In [ ]:

import requests
import time
import datetime 
import geopandas as gpd
import pandas as pd
from io import StringIO

map_key = "ea49e5fe6beaf3a00954c71727386596"

# Define Dates: start and enddate format (YYYY-MM-DD), oldest date 2017?
#startdate = "2024-09-22"
#enddate = "2024-09-30"

# calculate days out of dates --> FIRMS uses a number of days in the API-call
#start = pd.to_datetime(startdate)
#end = pd.to_datetime(enddate)

days = 5

# transform dates to FIRMS accepted format (YYYY-MM-DD)

# for loop später

#area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/world/1'

#date.today() fetches the current date -> is a date-object
# srftime converts dateobject to strings as FIRMS needs it in a string format (YYYY-MM-DD)

# source for this???


# define coordinates for South America  (min_lon, min_lat, max_lon, max_lat

region_coords = "-81.5,-55.0,-35.0,12.5"

# define sensor and products
sensor = "VIIRS_NOAA20_NRT"
product = "fire"

# define API endpoint url

api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{sensor}/{region_coords}/{days}"


# fetch the data

response = requests.get(api_url)

# check the status and extract the data
if response.status_code == 200: 
    print("Request successful\n")

    # read url and create dataframe
    df_fire = pd.read_csv(StringIO(response.text)) # for only 1 api request instead of 2

    # sample only 1000 / 500 rows; so my computer does not crash and I can develop my map
    if len(df_fire) > 500:
        df_fire = df_fire.sample(500, random_state=42)

    print(f"\n Data download successful! {len(df_fire)} fires found")  ## NA handling??
    display(df_fire.head(6))

    # option: include a saving option if wanted
    #df_fire.to_csv("firms_data_startdate_enddate.csv", index=False) wieso index = false??

elif response.status_code == 404:
    print(f"Error 404: Page not found")

elif response.status_code == 401:
    print(f"Error 401: Invalid API-Key, please check map-key")

elif response.status_code == 400:
    print(f"Error 400: Wrong parameters, please check input parameters.")


else: 
    print(f"Request failed: Status {response.status_code}")
    print(response.text)





Request successful


 Data download successful! 1000 fires found


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
7732,-12.28561,-44.63149,349.03,0.67,0.73,2026-05-09,1541,N20,VIIRS,n,2.0NRT,294.72,61.98,D
2464,-14.86031,-48.55337,324.21,0.50,0.66,2026-05-07,347,N20,VIIRS,n,2.0NRT,290.69,3.74,N
9515,-36.41210,-72.64176,326.99,0.44,0.39,2026-05-09,1858,N20,VIIRS,n,2.0NRT,295.71,11.25,D
3379,-10.24314,-47.41845,342.53,0.43,0.46,2026-05-07,1620,N20,VIIRS,n,2.0NRT,302.71,6.87,D
5615,-13.11477,-63.97070,332.01,0.43,0.38,2026-05-08,1741,N20,VIIRS,n,2.0NRT,296.99,4.53,D
9383,10.52693,-61.36925,347.54,0.39,0.36,2026-05-09,1730,N20,VIIRS,n,2.0NRT,308.00,6.87,D


: 

: 

notes:
with coordinates must be  min_lon, min_lat, max_lon, max_lat
sensor: "VIIRS_NOAA20_SP" does not give any data for timerange 5, with 20 it throws an Error 400

sensor: ""VIIRS_NOAA20_NRT" gives data for the last 5 days, 6 and 7 days are too many data input points

will work with the data collected for 5 days, if time write code so 7 days can be fetched (loop or sth.)

need to include a sample maximum of 1000 or so, to not overwhelm my computer, will take this out for the final map

#### Visualisation

##### Base Map

in this code, later a custom tile url could be included, if time allows

In [49]:
# convert csv to geodataframe
# FIRMS data are in crs EPSG:4326, no .to_crs() needed
# follium requires EPSG:4326

import folium


gdf_fire = gpd.GeoDataFrame(
    df_fire, 
    geometry=gpd.points_from_xy(
        df_fire.longitude, df_fire.latitude),
        
        crs="EPSG:4326")


## initialize basemap centered on South America
# folium location format: [lat, lon]
# later here could a custom tile url be added 
sa_map = folium.Map(location = [-15, -60], zoom_start = 3, tiles = "CartoDB DarkMatter"
)

# save map because I cannot display it in the notebook
sa_map.save("sa_map.html")

# display map
#sa_map

##### Fire Map Last 5 Days as Points

In [ ]:
# convert gdf to GEoJSON, then add to map as layer

fire_map1 = folium.GeoJson(gdf_fire.to_json()).add_to(sa_map)

#fire_map1.add_child(folium.Popup("Fire detected!"))

fire_map1.save("fire_map1.html")



now I have map with 1000 data points 
they are displayed as blue teardrop markers; want to change this to heat points or sth,

 maybe make a heat map later
 or sth to toggle on and off features (eg. sample for brightness / intensity, make groups or sth)

In [ ]:
# custom icons
# with tooltip to show intensity
# maybe convert this to popups later


fire_map2 = folium.GeoJson(fire_map1, 
               name = "Fire Detections", 
               tooltip = folium.GeoJsonTooltip(fields=["bright_ti4"], aliases=["Brightness:"]),
               fire_icon = folium.Icon(color = "orange", icon ="fire", prefix = "fa"), #fa = use font awesome icons, see https://fontawesome.com/icons/fire?s=solid
).add_to(sa_map)


fire_icon = folium.Icon(color="red", icon="fire", prefix="fa")

# add interactive layer control menu
folium.LayerControl().add_to(sa_map)

fire_map2.save("fire_map2.html")